## Bankruptcy Prediction using Multi-Seed LGBM–XGBoost Ensemble


### Author - Prerna Jha





## 1. Project Overview

This project aims to predict the probability of corporate bankruptcy using advanced machine-learning techniques. We build a high-performance two-model ensemble using LightGBM and XGBoost, enhanced with extensive feature engineering, quantile clipping, and multi-seed cross-validation. The final model uses weighted blending to achieve strong generalization and high AUC performance.

## 2. Dataset Description

The training dataset contains financial attributes and a binary target variable class, indicating whether a company is bankrupt. The test dataset includes only the features and an ID column. Because the dataset is imbalanced, strategies such as scale_pos_weight, stratified cross-validation, and robust preprocessing are used to stabilize training.

## 3. Preprocessing Pipeline

Quantile clipping (1%–99%) is applied to reduce the impact of extreme outliers.
For LightGBM, we generate additional statistical features including row_mean, row_std, row_max, row_min, along with log1p-transformations, squared terms, and ratios to row means.
For XGBoost, we use log1p-transformations and RobustScaler to create a stable representation without heavy feature expansion.

## 4. Model Architecture

Two independent models are trained:

LightGBM with 900 trees, moderate learning rate, subsampling, and full-depth trees.

XGBoost with 800 trees, max_depth=5, scale_pos_weight for imbalance handling, subsampling, and histogram tree method.
These models capture complementary patterns in the data, forming the basis of the ensemble.

## 5. Cross-Validation Strategy

We use 10-fold stratified K-fold cross-validation repeated over 5 different random seeds (42, 777, 30251, 123, 2024). This results in 100 total model fits and produces stable out-of-fold predictions for both models. The multi-seed approach significantly reduces variance and improves reliability.

## 6. Ensemble Blending

After collecting out-of-fold predictions, we perform a linear weight search over XGBoost and LightGBM outputs. We evaluate weights from 0 to 1 in steps of 0.025 and select the combination that yields the highest blended AUC. The final test predictions use this optimal weighted blend of the two model families.

## 7. Submission Files Generated

The script generates six different submission files to improve robustness:

Main weighted blend

Pure LightGBM predictions

Pure XGBoost predictions

Equal-weight (0.5–0.5) ensemble

LGBM-dominant blend

XGB-dominant blend
These alternatives help mitigate leaderboard sensitivity and distribution shifts.

## 8. Summary of Results

The code prints the optimal XGB/LGBM weights, blended AUC score, and prediction summary statistics including mean, standard deviation, minimum, and maximum values. A sample of the final submission file is also displayed for verification.

## 9. Technical Stack

Python, NumPy, Pandas, scikit-learn, LightGBM, XGBoost.
Stratified K-Fold is used for validation, RobustScaler for preprocessing, and CSV outputs for submissions.

## 10. How to Run

Place the training and test CSV files in the working directory, update file paths as needed, and run the script end-to-end. The output submission files are automatically saved to disk.

## 11. Key Strengths of the Approach

Strong outlier control, rich feature engineering, complementary model architectures, multi-seed validation for stability, and AUC-optimized blending. The combination of these techniques delivers a high-accuracy bankruptcy prediction system.

## 12. Conclusion

This project demonstrates an effective and systematic approach to bankruptcy prediction using a two-model ensemble. By combining LightGBM’s feature-rich boosting with XGBoost’s scaled, regularized trees, and optimizing their blend through multi-seed validation, the model achieves strong predictive performance and robust generalization.

In [1]:

import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import RobustScaler
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
import warnings
warnings.filterwarnings('ignore')

# =========================================================
# Paths
# =========================================================
TRAIN_PATH = "/content/bankruptcy_Train.csv"
TEST_PATH  = "/content/bankruptcy_Test_X.csv"
OUT_PATH   = "/content/submission_V28_MINIMAL.csv"

# =========================================================
# 1. Load data
# =========================================================
train = pd.read_csv(TRAIN_PATH)
test  = pd.read_csv(TEST_PATH)

y = train["class"].values
X = train.drop(columns=["class"])
X_test = test.drop(columns=["ID"])
test_ids = test["ID"].values

print("Train shape:", X.shape)
print("Test shape :", X_test.shape)

pos_rate = y.mean()
scale_pos = (1 - pos_rate) / pos_rate
print("Positive class rate:", pos_rate)
print("scale_pos_weight   :", scale_pos)

# =========================================================
# 2. Quantile clipping
# =========================================================
def quantile_clip(train_df: pd.DataFrame, test_df: pd.DataFrame,
                  q_low=0.01, q_high=0.99) -> (pd.DataFrame, pd.DataFrame):
    train_df = train_df.copy()
    test_df = test_df.copy()
    for col in train_df.columns:
        lo = train_df[col].quantile(q_low)
        hi = train_df[col].quantile(q_high)
        train_df[col] = train_df[col].clip(lo, hi)
        test_df[col]  = test_df[col].clip(lo, hi)
    return train_df, test_df

X_clip, X_test_clip = quantile_clip(X, X_test, q_low=0.01, q_high=0.99)
print("After clipping - train:", X_clip.shape)
print("After clipping - test :", X_test_clip.shape)

# =========================================================
# 3A. Feature engineering for LGBM
# =========================================================
def fe_light(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    cols = df.columns.tolist()
    eps = 1e-6

    row_mean = df[cols].mean(axis=1)
    row_std  = df[cols].std(axis=1)
    row_max  = df[cols].max(axis=1)
    row_min  = df[cols].min(axis=1)

    df["row_mean"] = row_mean
    df["row_std"]  = row_std
    df["row_max"]  = row_max
    df["row_min"]  = row_min

    for c in cols:
        df[f"log1p_{c}"] = np.log1p(np.abs(df[c]))
        df[f"{c}_sq"]    = df[c] ** 2
        df[f"{c}_div_rowmean"] = df[c] / (row_mean + eps)

    df.replace([np.inf, -np.inf], 0, inplace=True)
    df.fillna(0, inplace=True)
    return df

X_all_clip = pd.concat([X_clip, X_test_clip], axis=0).reset_index(drop=True)
X_all_fe = fe_light(X_all_clip)

X_lgb_train = X_all_fe.iloc[:len(X)].reset_index(drop=True).astype("float32").values
X_lgb_test  = X_all_fe.iloc[len(X):].reset_index(drop=True).astype("float32").values

print("LGBM FE train shape:", X_lgb_train.shape)
print("LGBM FE test  shape:", X_lgb_test.shape)

# =========================================================
# 3B. Enhanced raw view for XGB
# =========================================================
X_raw_enh = X_clip.copy()
X_test_raw_enh = X_test_clip.copy()

for c in X_raw_enh.columns:
    X_raw_enh[f"log1p_{c}"] = np.log1p(np.abs(X_raw_enh[c]))
    X_test_raw_enh[f"log1p_{c}"] = np.log1p(np.abs(X_test_raw_enh[c]))

scaler = RobustScaler()
X_raw_train = scaler.fit_transform(X_raw_enh.astype("float32"))
X_raw_test  = scaler.transform(X_test_raw_enh.astype("float32"))

print("XGB raw-enh train shape:", X_raw_train.shape)
print("XGB raw-enh test  shape:", X_raw_test.shape)

y_arr = y

# =========================================================
# 4. ONLY CHANGE: More seeds for stability (3→5)
# =========================================================
SEEDS = [42, 777, 30251, 123, 2024]  # Added 2 more seeds
N_FOLDS = 10

oof_lgb_all = np.zeros(len(y_arr))
oof_xgb_all = np.zeros(len(y_arr))
test_lgb_all = np.zeros(len(X_lgb_test))
test_xgb_all = np.zeros(len(X_raw_test))

for seed in SEEDS:
    print(f"\n================= SEED {seed} =================")
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=seed)

    oof_lgb_seed = np.zeros(len(y_arr))
    oof_xgb_seed = np.zeros(len(y_arr))
    test_lgb_seed = np.zeros(len(X_lgb_test))
    test_xgb_seed = np.zeros(len(X_raw_test))

    fold_id = 1
    for tr_idx, val_idx in skf.split(X_lgb_train, y_arr):
        X_tr_lgb, X_val_lgb = X_lgb_train[tr_idx], X_lgb_train[val_idx]
        X_tr_raw, X_val_raw = X_raw_train[tr_idx], X_raw_train[val_idx]
        y_tr, y_val = y_arr[tr_idx], y_arr[val_idx]

        # LGBM - EXACT V23 hyperparameters
        lgb_model = LGBMClassifier(
            n_estimators=900,
            max_depth=-1,
            learning_rate=0.03,
            subsample=0.9,
            colsample_bytree=0.8,
            objective="binary",
            reg_lambda=1.0,
            random_state=seed + fold_id,
            n_jobs=-1
        )
        lgb_model.fit(X_tr_lgb, y_tr)
        val_lgb = lgb_model.predict_proba(X_val_lgb)[:, 1]
        oof_lgb_seed[val_idx] = val_lgb
        test_lgb_seed += lgb_model.predict_proba(X_lgb_test)[:, 1] / N_FOLDS

        # XGB - EXACT V23 hyperparameters
        xgb_model = XGBClassifier(
            n_estimators=800,
            max_depth=5,
            learning_rate=0.03,
            subsample=0.9,
            colsample_bytree=0.8,
            objective="binary:logistic",
            eval_metric="auc",
            reg_lambda=1.0,
            reg_alpha=0.0,
            scale_pos_weight=scale_pos,
            tree_method="hist",
            random_state=seed + fold_id,
            n_jobs=-1
        )
        xgb_model.fit(X_tr_raw, y_tr)
        val_xgb = xgb_model.predict_proba(X_val_raw)[:, 1]
        oof_xgb_seed[val_idx] = val_xgb
        test_xgb_seed += xgb_model.predict_proba(X_raw_test)[:, 1] / N_FOLDS

        print(f"  Seed {seed} | Fold {fold_id} AUCs -> "
              f"LGB: {roc_auc_score(y_val, val_lgb):.5f} | "
              f"XGB: {roc_auc_score(y_val, val_xgb):.5f}")
        fold_id += 1

    print(f"Seed {seed} full OOF AUCs: LGBM={roc_auc_score(y_arr, oof_lgb_seed):.5f} | "
          f"XGB={roc_auc_score(y_arr, oof_xgb_seed):.5f}")

    oof_lgb_all += oof_lgb_seed / len(SEEDS)
    oof_xgb_all += oof_xgb_seed / len(SEEDS)
    test_lgb_all += test_lgb_seed / len(SEEDS)
    test_xgb_all += test_xgb_seed / len(SEEDS)

# =========================================================
# 5. 2-model weight search
# =================
auc_lgb = roc_auc_score(y_arr, oof_lgb_all)
auc_xgb = roc_auc_score(y_arr, oof_xgb_all)
print("\n==== Multi-seed Base Model OOF AUCs ====")
print(f"LGBM OOF AUC: {auc_lgb:.5f}")
print(f"XGB  OOF AUC: {auc_xgb:.5f}")

weights = np.linspace(0, 1, 41)
best_auc = 0.0
best_w = None

for w_xgb in weights:
    w_lgb = 1.0 - w_xgb
    blend_oof = w_xgb * oof_xgb_all + w_lgb * oof_lgb_all
    auc = roc_auc_score(y_arr, blend_oof)
    if auc > best_auc:
        best_auc = auc
        best_w = (w_xgb, w_lgb)

print("\nBest 2-model blend weights (XGB, LGBM):", best_w)
print("Best blended OOF AUC:", round(best_auc, 5))

w_xgb, w_lgb = best_w
final_test_pred = w_xgb * test_xgb_all + w_lgb * test_lgb_all

# =========================================================
# 6. ADDITIONAL SUBMISSIONS: Try slight variations
# =========================================================

# Submission 1: Exact weighted blend (main)
submission_main = pd.DataFrame({
    "ID": test_ids,
    "class": final_test_pred
})
submission_main.to_csv(OUT_PATH, index=False)
print(f"\n[MAIN] Saved: {OUT_PATH}")

# Submission 2: Pure LGBM (in case ensemble hurts)
submission_lgbm = pd.DataFrame({
    "ID": test_ids,
    "class": test_lgb_all
})
submission_lgbm.to_csv(OUT_PATH.replace('.csv', '_lgbm_only.csv'), index=False)
print(f"[ALT1] Saved: {OUT_PATH.replace('.csv', '_lgbm_only.csv')}")

# Submission 3: Pure XGB (in case ensemble hurts)
submission_xgb = pd.DataFrame({
    "ID": test_ids,
    "class": test_xgb_all
})
submission_xgb.to_csv(OUT_PATH.replace('.csv', '_xgb_only.csv'), index=False)
print(f"[ALT2] Saved: {OUT_PATH.replace('.csv', '_xgb_only.csv')}")

# Submission 4: Equal weights (0.5, 0.5)
submission_equal = pd.DataFrame({
    "ID": test_ids,
    "class": 0.5 * test_lgb_all + 0.5 * test_xgb_all
})
submission_equal.to_csv(OUT_PATH.replace('.csv', '_equal.csv'), index=False)
print(f"[ALT3] Saved: {OUT_PATH.replace('.csv', '_equal.csv')}")

# Submission 5: Slightly more LGBM than optimal (if optimal is close to edge)
w_lgb_plus = min(w_lgb + 0.05, 1.0)
w_xgb_minus = 1.0 - w_lgb_plus
submission_lgbm_plus = pd.DataFrame({
    "ID": test_ids,
    "class": w_xgb_minus * test_xgb_all + w_lgb_plus * test_lgb_all
})
submission_lgbm_plus.to_csv(OUT_PATH.replace('.csv', '_lgbm_plus.csv'), index=False)
print(f"[ALT4] Saved: {OUT_PATH.replace('.csv', '_lgbm_plus.csv')}")

# Submission 6: Slightly more XGB than optimal
w_xgb_plus = min(w_xgb + 0.05, 1.0)
w_lgb_minus = 1.0 - w_xgb_plus
submission_xgb_plus = pd.DataFrame({
    "ID": test_ids,
    "class": w_xgb_plus * test_xgb_all + w_lgb_minus * test_lgb_all
})
submission_xgb_plus.to_csv(OUT_PATH.replace('.csv', '_xgb_plus.csv'), index=False)
print(f"[ALT5] Saved: {OUT_PATH.replace('.csv', '_xgb_plus.csv')}")

print("\n" + "="*60)
print("SUMMARY")
print("="*60)
print(f"Optimal weights: XGB={w_xgb:.3f}, LGBM={w_lgb:.3f}")
print(f"Prediction stats (main):")
print(f"  Mean: {final_test_pred.mean():.5f}")
print(f"  Std:  {final_test_pred.std():.5f}")
print(f"  Min:  {final_test_pred.min():.5f}")
print(f"  Max:  {final_test_pred.max():.5f}")
print(f"\nAll Model run complete")
print(submission_main.head())

Train shape: (10000, 64)
Test shape : (8000, 64)
Positive class rate: 0.0473
scale_pos_weight   : 20.14164904862579
After clipping - train: (10000, 64)
After clipping - test : (8000, 64)
LGBM FE train shape: (10000, 260)
LGBM FE test  shape: (8000, 260)
XGB raw-enh train shape: (10000, 128)
XGB raw-enh test  shape: (8000, 128)

================= SEED 42 =================
[LightGBM] [Info] Number of positive: 426, number of negative: 8574
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.033770 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 64114
[LightGBM] [Info] Number of data points in the train set: 9000, number of used features: 260
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.047333 -> initscore=-3.002050
[LightGBM] [Info] Start training from score -3.002050
  Seed 42 | Fold 1 AUCs -> LGB: 0.92416 | XGB: 0.91893
[LightGBM] [Info] Number of positive: 426, number of negative: 8574
[LightGBM] 